In [1]:
import requests
from dotenv import load_dotenv
import os


load_dotenv()

def medical_chat_completion(
    message: str,
    model: str = "medgemma-4b-it",
    max_tokens: int = 1000,
    temperature: float = 0.7,
):
    url = "https://dr7.ai/api/v1/medical/chat/completions"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.environ['DR7_API_KEY']}",
    }

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": message,
            }
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()  # Raises an exception for 4xx/5xx responses

    return response.json()

In [3]:
result = medical_chat_completion(
    message="Patient reports headache for 3 days with fever of 38.5°C"
)

print(result)

HTTPError: 402 Client Error: Payment Required for url: https://dr7.ai/api/v1/medical/chat/completions

In [1]:
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os


load_dotenv()


# 1. Initialize the client with your free access token
# Replace 'your_hf_token_here' with your actual token string

HF_TOKEN = os.environ['HF_TOKEN']
# 2. Initialize the client using standard architecture
client = InferenceClient(provider="featherless-ai", token=HF_TOKEN)

# 3. Target a universally supported free-tier model
# Qwen2.5 and Llama 3.1 are permanently deployed on HF serverless infrastructure
model_id = "google/medgemma-27b-text-it"

messages = [
    {
        "role": "system",
        "content": (
            "You are an advanced clinical AI assistant. Analyze the symptoms provided, "
            "offer structured differential considerations, and state standard medical triage steps. "
            "Always append a clear disclaimer that you are an AI and not a substitute for a real doctor."
        )
    },
    {
        "role": "user",
        "content": "Patient reports headache for 3 days with a fever of 38.5°C."
    }
]

print(f"Streaming response from {model_id}...\n")

try:
    # Use client.chat_completion with stream=True
    # This natively yields simple chunk objects that contain straight text strings
    response_stream = client.chat_completion(
        model=model_id,
        messages=messages,
        max_tokens=450,
        stream=True
    )

    for chunk in response_stream:
        # Pull text safely from the choices array dictionary format
        if hasattr(chunk, 'choices') and chunk.choices:
            content = chunk.choices[0].delta.content
            if content:
                print(content, end="", flush=True)
        # Fallback if the provider returns a direct dictionary
        elif isinstance(chunk, dict) and 'choices' in chunk:
            content = chunk['choices'][0].get('delta', {}).get('content', '')
            if content:
                print(content, end="", flush=True)

except Exception as e:
    print(f"\nAn error occurred: {e}")

Streaming response from google/medgemma-27b-text-it...

Okay, I can analyze the provided symptoms and offer a structured differential diagnosis and triage recommendations based on standard medical practice.

**Patient Symptoms:**

*   Headache (3 days duration)
*   Fever (38.5°C / 101.3°F)

**Analysis & Differential Considerations:**

The combination of headache and fever suggests an underlying inflammatory or infectious process. The duration of 3 days is subacute, which helps narrow the possibilities somewhat but still leaves a broad range. Here's a structured approach to differential considerations, categorized by potential severity and system involvement:

**I. High Priority / Potentially Life-Threatening:**

*   **Meningitis (Bacterial or Viral):**
    *   *Rationale:* Headache and fever are classic signs. Bacterial meningitis is a medical emergency. Viral meningitis is generally less severe but still requires evaluation.
    *   *Associated Symptoms to Elicit:* Stiff neck (nuchal 

In [2]:
"""
query_space.py  -  call the deployed Hybrid EHR model on Hugging Face Spaces,
building a SENSIBLE dummy patient from the model's own feature means.

    pip install requests
    python query_space.py

Requires the /metadata endpoint added to server.py (redeploy the Space after adding it).
"""
import time
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# Direct app URL (*.hf.space) - confirm via Space page -> ⋮ -> "Embed this Space".
BASE_URL = "https://victorano-healthpilot-hybrid-model.hf.space"
API_TOKEN = os.environ['INFERENCE_API_TOKEN']   # only if INFERENCE_API_TOKEN is set as a Space secret

# Override specific feature columns by their EXACT name (printed below) to model a
# patient. Anything not listed stays at the population mean (a neutral baseline).
# Fill these in AFTER you see the real column names from /metadata, e.g.:
#   OVERRIDES = {"hba1c_mean": 9.2, "glucose_last": 180, "bp_systolic_mean": 152}
OVERRIDES: dict[str, float] = {}

DUMMY_NOTE = (
    "S: 58-year-old female, PMH type 2 diabetes mellitus, hypertension, and "
    "hyperlipidemia. Routine follow-up; reports increased fatigue and occasional "
    "blurred vision over the past month. Denies chest pain or dyspnea. "
    "O: BP 152/94, HR 78, BMI 33.1. HbA1c 8.4%, fasting glucose 178 mg/dL, "
    "LDL 142 mg/dL, eGFR 68. Trace bilateral lower-extremity edema. NKDA. "
    "Meds: metformin 1000mg BID, lisinopril 20mg daily, atorvastatin 40mg daily. "
    "A: Suboptimally controlled T2DM with possible early diabetic nephropathy; "
    "uncontrolled HTN; hyperlipidemia. "
    "P: Up-titrate lisinopril, add empagliflozin, dietary counseling, recheck labs "
    "and urine albumin in 3 months, ophthalmology referral for retinopathy screening."
)


def headers():
    h = {"Content-Type": "application/json"}
    if API_TOKEN:
        h["Authorization"] = f"Bearer {API_TOKEN}"
    return h


def wait_until_awake(timeout=300, interval=8):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            r = requests.get(f"{BASE_URL}/health", timeout=20)
            if r.ok and r.json().get("loaded"):
                print("Space is awake and model is loaded.")
                return
            print(f"  waking up... ({r.status_code})")
        except requests.RequestException as e:
            print(f"  waiting for cold start... ({type(e).__name__})")
        time.sleep(interval)
    raise TimeoutError("Space did not become ready in time.")


def get_metadata():
    r = requests.get(f"{BASE_URL}/metadata", headers=headers(), timeout=30)
    r.raise_for_status()
    return r.json()


def predict(features, note, top_k=None):
    payload = {"features": features, "note": note, "top_k": top_k}
    r = requests.post(f"{BASE_URL}/predict", json=payload,
                      headers=headers(), timeout=120)
    r.raise_for_status()
    return r.json()


def print_predictions(result):
    print(f"\n{'prob':>7}  {'thr':>5}  pred  condition")
    print("-" * 60)
    for c in result["conditions"]:
        flag = "  <== flagged" if c["predicted"] else ""
        print(f"{c['probability']:7.4f}  {c['threshold']:5.2f}   {c['predicted']}    "
              f"{c['label']}{flag}")


if __name__ == "__main__":
    wait_until_awake()
    meta = get_metadata()
    cols = meta["feature_columns"]
    means = dict(zip(cols, meta["feature_means"]))

    print(f"\nModel expects {meta['n_features']} features. Column names:")
    for i, c in enumerate(cols):
        print(f"  [{i:>2}] {c}  (mean={means[c]:.3g})")

    # Neutral patient = population means; apply any overrides by exact column name.
    patient = dict(means)
    for k, v in OVERRIDES.items():
        if k not in patient:
            raise KeyError(f"Override '{k}' is not a real feature column.")
        patient[k] = v

    features = [patient[c] for c in cols]   # exact training order
    result = predict(features, DUMMY_NOTE, top_k=10)
    print_predictions(result)

Space is awake and model is loaded.

Model expects 80 features. Column names:
  [ 0] AGE  (mean=45.5)
  [ 1] active_careplan_count  (mean=3.95)
  [ 2] total_prescriptions  (mean=4.34)
  [ 3] active_medication_count  (mean=1.91)
  [ 4] enc_encounter_for_symptom  (mean=2.07)
  [ 5] Body Height_hist_mean  (mean=124)
  [ 6] Systolic Blood Pressure_hist_std  (mean=10.4)
  [ 7] total_encounters  (mean=14.6)
  [ 8] Diastolic Blood Pressure_hist_std  (mean=5.11)
  [ 9] Diastolic Blood Pressure_hist_mean  (mean=68)
  [10] Systolic Blood Pressure_hist_mean  (mean=105)
  [11] unique_procedure_types  (mean=2.6)
  [12] Body Weight_hist_std  (mean=3.97)
  [13] Body Mass Index_hist_std  (mean=0.968)
  [14] rx_analgesic  (mean=0.578)
  [15] Body Mass Index_hist_mean  (mean=22.2)
  [16] total_procedures  (mean=7.31)
  [17] Body Height_hist_std  (mean=3.61)
  [18] Body Weight_hist_mean  (mean=59.3)
  [19] Systolic Blood Pressure_hist_last  (mean=105)
  [20] High Density Lipoprotein Cholesterol_hist_std 

In [ ]:
inference_api_token=